# Multi-GPU Vegetation Risk Vision Pipeline - Colab Run

This notebook reproduces the VEPL real-UAV segmentation experiment on Google Colab. Use `Runtime > Change runtime type > T4 GPU` before running.

Colab usually exposes one GPU, so this notebook validates the single-GPU CUDA path. Use Kaggle T4x2 for the DDP two-GPU path.

In [ ]:
!nvidia-smi

import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

## Clone The Project

The notebook always starts from the GitHub version so the run is reproducible from a clean Colab runtime.

In [ ]:
%cd /content
!rm -rf multi-gpu-vegetation-risk-vision-pipeline
!git clone https://github.com/msa-1988/multi-gpu-vegetation-risk-vision-pipeline.git
%cd /content/multi-gpu-vegetation-risk-vision-pipeline

## Install Lightweight Dependencies

Colab already provides PyTorch and torchvision. Installing only the lightweight packages avoids replacing the CUDA-enabled Colab torch build.

In [ ]:
!pip install -q numpy tqdm matplotlib

import os
os.environ['PYTHONPATH'] = '/content/multi-gpu-vegetation-risk-vision-pipeline/src'

## Download VEPL Real UAV Tiles

This downloads the compact non-augmented VEPL archive from Zenodo. It is about 77 MB.

In [ ]:
!bash scripts/download_vepl_sample.sh
!PYTHONPATH=src python scripts/visualize_vepl_samples.py --samples 4 --image-size 256 --output artifacts/figures/colab_vepl_samples.png

In [ ]:
from IPython.display import Image, display
display(Image('artifacts/figures/colab_vepl_samples.png'))

## Fast Colab GPU Run

Run this first. It uses the full compact VEPL split but fewer epochs than the README benchmark, so you can verify the end-to-end workflow quickly.

In [ ]:
!PYTHONPATH=src python -m veg_multigpu.train \
  --dataset vepl \
  --data-dir data/vepl/TESELLATED_WITHOUT_AUGMENTATION \
  --vepl-target foreground \
  --epochs 8 \
  --batch-size 8 \
  --image-size 192 \
  --train-samples 426 \
  --val-samples 106 \
  --base-channels 32 \
  --num-workers 2 \
  --amp \
  --device cuda \
  --out-dir artifacts/runs/colab_vepl_quick

## Inspect Metrics And Predictions

In [ ]:
!PYTHONPATH=src python scripts/compare_runs.py
!PYTHONPATH=src python scripts/plot_training_curves.py --metrics artifacts/runs/colab_vepl_quick/metrics.json --output artifacts/figures/colab_training_curves.png
!PYTHONPATH=src python scripts/visualize_vepl_predictions.py \
  --checkpoint artifacts/runs/colab_vepl_quick/best_model.pt \
  --output artifacts/figures/colab_vepl_predictions.png \
  --image-size 192 \
  --base-channels 32 \
  --samples 4 \
  --device cuda

display(Image('artifacts/figures/colab_training_curves.png'))
display(Image('artifacts/figures/colab_vepl_predictions.png'))

## Optional Full README-Style Run

Set `RUN_FULL = True` to run the 40-epoch experiment used for the README benchmark. It saves `best_model.pt` by validation IoU.

In [ ]:
RUN_FULL = False

if RUN_FULL:
    !PYTHONPATH=src python -m veg_multigpu.train \
      --dataset vepl \
      --data-dir data/vepl/TESELLATED_WITHOUT_AUGMENTATION \
      --vepl-target foreground \
      --epochs 40 \
      --batch-size 8 \
      --image-size 192 \
      --train-samples 426 \
      --val-samples 106 \
      --base-channels 32 \
      --num-workers 2 \
      --amp \
      --device cuda \
      --out-dir artifacts/runs/colab_vepl_full
    !PYTHONPATH=src python scripts/compare_runs.py